In [1]:
import pandas as pd
import numpy as np
import os
import glob as glob
import sys
import json

parent_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(parent_dir)
import cpg_harmonizer
import s3_loader
import harmonize_checker

In [2]:
%load_ext autoreload
%autoreload 2

# Enter Identifiers, Per-Dataset Metadata

In [3]:
in_staging = True # True or False
profile = "CPGnew" # enter AWS profile if in_staging is True, otherwise None

In [4]:
project_name = "cpg0049-tcf7l2-pertubation"
# NOTE: PROJECT NAME WILL BE CHANGED TO "cpg0049-ipsc-diff-pancreatic-progenitor" when transferred to production
source_list = ['nhgri'] # e.g. ['broad'] or ['source_1','source_2']
output_parent_directory = "/Users/eweisbar/Desktop/cpg0049metadata"

# typically, per-dataset metadata
# if not dataset-wide, delete from here and create conditional entry below
# if unknown, comment out
per_dataset_manual = {
    'Plate_Size':384,
    'CP_Version':'v2', # e.g. "v1"
    'DOI_to_Cite':'',
    'Year_Imaged': 2025,
    'Cell_Line_Name':'None',
    'Cell_Line_Organism':'Homo sapiens',
    #'Cell_Line_Type' - per pertubation
    #'Cell_Line_Modification' - per perturbation
    'Microscope_Name':'Revvity Opera Phenix',
    'Microscope_Binning': 1,
    'Microscope_Modality':'Confocal', # e.g. 'Widefield
    'Microscope_Objective_Magnification':40, # e.g. 20
    'Microscope_Objective_NA':1.1, # e.g. .45
    'Microscope_Pixel_Size': .15,
    'Image_Bit_Depth': 16, # e.g. 16
    'Image_Size_X':2160,
    'Image_Size_Y':2160,
    #'Timepoint_Primary_Treatment':0,
    #'Timepoint_Secondary_Treatment':0,
    #'Timepoint_Acquisition':0,
    'Treatment_Category':'None', #e.g. 'Compound'
    'Treatment_Primary_Treatment': 'None', # e.g. 'Compound'
}

In [5]:
# excitation/emission values used for acquisition
# if value is unknown, define value with np.nan
# if excitation is with a laser (negligible width), 'ex_width' = "None"

# 'fluorophore' is fluorophore conjugated to label or dye variant
# "None" if none, np.nan if unknown
ex_em_fluor_dict = {
    'ER':{'ex_peak':488,
           'ex_width':np.nan,
           'em_peak':522,
           'em_width':np.nan,
           'fluorophore':'Alexa Fluor 488'},
    'Mito':{'ex_peak':640,
          'ex_width':np.nan,
           'em_peak':706,
           'em_width':np.nan,
           'fluorophore':'Deep Red'},
    'AGP':{'ex_peak':561,
           'ex_width':np.nan,
           'em_peak':599,
           'em_width':np.nan,
           'fluorophore':'Alexa Fluor 568'},
    'DNA':{'ex_peak':375,
           'ex_width':np.nan,
           'em_peak':456,
           'em_width':np.nan,
           'fluorophore':'33342'}
}

# Join CPG Metadata

In [6]:
if len(source_list) == 1:
    output_directory = os.path.join(output_parent_directory, project_name, source_list[0], "workspace", "metadata_harmonized")
else:
    output_directory = os.path.join(output_parent_directory, project_name, "all", "workspace", "metadata_harmonized")
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=True)

metadata_paths = []
load_paths = []
for src in source_list:
    metadata_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/metadata/platemaps/", in_staging=in_staging, profile=profile))
    load_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/load_data_csv/", in_staging=in_staging, profile=profile))

project = cpg_harmonizer.Project(output_directory, "../output_structure.json", project_name=project_name)

In [7]:
load_data_csvs = []
list_batch_names_from_load_data = []
for path in load_paths:
    if path.endswith("load_data.csv"):
        load_data_csvs.append(s3_loader.read_s3_file(path, sep = ",", in_staging=in_staging, profile=profile))
        list_batch_names_from_load_data.append(path.split("/")[-3])
if len(load_data_csvs) == 0:
    print("No load_data.csv files found. Please check the load_data_csv folder.")

In [8]:
platemaps = []
barcode_platemap_csvs = []
list_batch_names_from_barcode_platemaps = []
list_platemap_names = []
list_batch_names_from_platemaps = []
external_tsv = []

for path in metadata_paths:
    if path.endswith("barcode_platemap.csv"):
        barcode_platemap_csvs.append(s3_loader.read_s3_file(path, sep = ",",in_staging=in_staging, profile=profile))
        list_batch_names_from_barcode_platemaps.append(path.split("/")[-2])
    if '/platemap/' in path and path.endswith(".txt"):
        platemaps.append(s3_loader.read_s3_file(path, sep = "\t", in_staging=in_staging, profile=profile))
        list_platemap_names.append(path.split("/")[-1].split(".")[0])
        list_batch_names_from_platemaps.append(path.split("/")[-3])
    if 'external' in path:
        if '.csv' in path:
            external_tsv.append(s3_loader.read_s3_file(path, sep = ",", in_staging=in_staging, profile=profile))
        if '.tsv' in path:
            external_tsv.append(s3_loader.read_s3_file(path, sep = "\t", in_staging=in_staging, profile=profile))
if barcode_platemap_csvs == []:
    print("No barcode_platemap.csv files found. Please check the metadata/platemaps folder.")
if platemaps == []:
    print("No platemap files found. Please check the metadata/platemaps folder.")

In [9]:
all_load_data_platenames = []
for df in load_data_csvs:
    all_load_data_platenames.extend(df['Metadata_Plate'].unique())
all_barcode_platemap_platenames = []
for df in barcode_platemap_csvs:
    all_barcode_platemap_platenames.extend(df['Plate_Map_Name'].unique())
if set(all_load_data_platenames) != set(all_barcode_platemap_platenames):
    print("Plate names in load_data.csv and barcode_platemap.csv do not match.")
    print('Do NOT proceed until they are matched')

In [10]:
if not list(set(list_batch_names_from_platemaps)) == list(set(list_batch_names_from_load_data)):
    print("Warning: Batch names in platemaps and load_data.csv do not match.")
    print(f"Batch names from platemaps: {list(set(list_batch_names_from_platemaps))}")
    print(f"Batch names from load_data.csv: {list(set(list_batch_names_from_load_data))}")
if not list(set(list_batch_names_from_platemaps)) == list(set(list_batch_names_from_barcode_platemaps)):
    print("Warning:Batch names in platemaps and barcode_platemap.csv do not match.")
    print(f"Batch names from platemaps: {list(set(list_batch_names_from_platemaps))}")
    print(f"Batch names from barcode_platemap.csv: {list(set(list_batch_names_from_barcode_platemaps))}")

In [23]:
complete_df = project.run_conversion(
    load_data_csvs = load_data_csvs, 
    load_data_csv_batch = list_batch_names_from_load_data, 
    platemap_csvs = barcode_platemap_csvs,  
    platemap_csv_batch = list_batch_names_from_barcode_platemaps, 
    platemap_txt= platemaps, 
    platemap_txt_batch = list_batch_names_from_platemaps,
    platemap_txt_name = list_platemap_names,
    external_tsv = external_tsv,
    external_merge_regex = [["compound"],["compound"]]
)

No external metadata found. Not all projects have external metadata.
Merging in barcode platemap csvs
Merging in platemaps
Skipping external metadata. Not all projects have external metadata.
No concentration columns found — skipping concentration merge.
Harmonizing values of Label


# Additional cleaning steps

In [24]:
# add per-experiment metadata manually annotated above
for col, val in per_dataset_manual.items():
    complete_df[col] = val

# add source information inferred from file path
for source in source_list:
    complete_df.loc[complete_df['File Path'].str.contains(f"/{source}/"),'Source'] = source

In [25]:
with open('../inferable_relationships.json', "r") as f:
    inferable_metadata = json.load(f)

# infer label metadata from known relationships
if per_dataset_manual['CP_Version'] == 'other':
    print("Did not infer label metadata because CP_Version not inferrable")
    print(f"Labels that need to be manually declared are {complete_df['Label'].unique()}")
else:
    found_mismatch = False
    if not all([x in inferable_metadata["Label"].keys() for x in complete_df['Label'].unique()]):
        for x in complete_df['Label'].unique():
            if x not in inferable_metadata["Label"].keys():
                for key, value in inferable_metadata["Label_Alternative_Names"].items():
                    if x in value:
                        print(f"Inferred label {x} to be {key} based on alternative names")
                        complete_df.loc[complete_df['Label'] == x, 'Label'] = key
                        break
                else:
                    # only if there is a label that can't be inferred
                    found_mismatch = True 
    if found_mismatch:
        print(f'Labels need to be corrected to match any of {list(inferable_metadata["Label"].keys())}')
        print(f"Current labels are {complete_df['Label'].unique()}")

# use inferable metadata to fill in missing metadata for each label and cell line
for column in ["Cell_Line_Name", "Label"]:
    for entry in inferable_metadata[column]:
        for inferred_column in inferable_metadata[column][entry]:
            if not inferred_column in complete_df.columns:
                complete_df[inferred_column] = np.nan
                complete_df[inferred_column] = complete_df[inferred_column].astype('str')
            complete_df.loc[complete_df[column]==entry, inferred_column] = inferable_metadata[column][entry][inferred_column]


Inferred label Glycoproteins to be ER based on alternative names
Inferred label Mitochondria to be Mito based on alternative names


In [26]:
# add per label excitation and emission values
# can delete if all values are unknown
if set(complete_df['Label'].unique()) == ex_em_fluor_dict.keys():
    for key in ex_em_fluor_dict:
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Peak"] = ex_em_fluor_dict[key]['ex_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Width"] = ex_em_fluor_dict[key]['ex_width']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Peak"] = ex_em_fluor_dict[key]['em_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Width"] = ex_em_fluor_dict[key]['em_width']
        complete_df.loc[complete_df["Label"] == key, "Label_Fluorophore"] = ex_em_fluor_dict[key]['fluorophore']
else:
    print("Excitation/Emission dictionary does not match Labels")
    print(f"Available labels are {set(complete_df['Label'].unique())}")
    print(f"Defined keys are {ex_em_fluor_dict.keys()}")

In [27]:
# dataset specific cleaning
complete_df['Image_Position_Z'] = complete_df['Z'].str.replace('p','')
complete_df['Site'] = complete_df['Site'].str.replace('f','')

In [28]:
# report on un-harmonized columns
# use reported information to manually update ontology OR dataframe OR input metadata
# this cell does NOT save the harmonization results, just check them
# this allows you to make any necessary corrections without accidentally overwriting
# returns a view of what the dataframe will look like after final harmonization
extra_cols = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df, ret="extra_cols")

print("View of data that will be kept")
complete_df[[x for x in complete_df.columns if x not in extra_cols]]

Removing columns not in ontology: ['Z', 'Label', 'PASS_LINE_QC', 'stage', 'Plate_Map', 'User_Name']
Adding missing ontology columns: ['Treatment_SMILES', 'Treatment_Broad_Sample', 'Treatment_InChIKey', 'Timepoint_Primary_Treatment', 'Treatment_Concentration', 'Treatment_Control_Class', 'Treatment_PubChem_CID', 'Timepoint_Secondary_Treatment', 'Timepoint_Acquisition', 'Treatment_Secondary_Treatment', 'Treatment_Solvent', 'Treatment_Mechanism']
View of data that will be kept


,Well,Site,Plate,Batch,File Path,File Name,Cell_Line_Type,Cell_Line_Modification,Plate_Size,CP_Version,...,Label_Reagent,Label_Structure,Label_Molecule,Label_Mechanism,Microscope_Excitation_Peak,Microscope_Excitation_Width,Microscope_Emission_Peak,Microscope_Emission_Width,Label_Fluorophore,Image_Position_Z
0,K20,13,NIHB111_C384W_201,2024_05_10_NIHB111_S1,https://cellpainting-gallery.s3.us-east-1.amaz...,r11c20f13p01-ch1sk1fk1fl1,iPSC differentiated to beta cells,"MODY Patient derived clonal line, HNF4A::c.925...",384,v2,...,ConcanavalinA (ConA),Endoplasmic Reticulum,Glycoproteins,Dye,488,<NA>,522,<NA>,Alexa Fluor 488,1
1,C14,7,NIHB111_C384W_201,2024_05_10_NIHB111_S1,https://cellpainting-gallery.s3.us-east-1.amaz...,r03c14f07p01-ch1sk1fk1fl1,iPSC differentiated to beta cells,"Clonal line, TCF7L2/HNF4A/HNF1A::WT/WT, 1-Defi...",384,v2,...,ConcanavalinA (ConA),Endoplasmic Reticulum,Glycoproteins,Dye,488,<NA>,522,<NA>,Alexa Fluor 488,1
2,G12,22,NIHB111_C384W_201,2024_05_10_NIHB111_S1,https://cellpainting-gallery.s3.us-east-1.amaz...,r07c12f22p01-ch1sk1fk1fl1,iPSC differentiated to beta cells,"Clonal line, TCF7L2/HNF4A/HNF1A::WT/WT, 1-Defi...",384,v2,...,ConcanavalinA (ConA),Endoplasmic Reticulum,Glycoproteins,Dye,488,<NA>,522,<NA>,Alexa Fluor 488,1
3,J04,11,NIHB111_C384W_201,2024_05_10_NIHB111_S1,https://cellpainting-gallery.s3.us-east-1.amaz...,r10c04f11p01-ch1sk1fk1fl1,iPSC differentiated to beta cells,"Clonal line, TCF7L2/HNF4A/HNF1A::WT/WT, 1-Defi...",384,v2,...,ConcanavalinA (ConA),Endoplasmic Reticulum,Glycoproteins,Dye,488,<NA>,522,<NA>,Alexa Fluor 488,1
4,E05,24,NIHB111_C384W_201,2024_05_10_NIHB111_S1,https://cellpainting-gallery.s3.us-east-1.amaz...,r05c05f24p01-ch1sk1fk1fl1,iPSC differentiated to beta cells,"MODY Patient derived clonal line, HNF1A::c.872...",384,v2,...,ConcanavalinA (ConA),Endoplasmic Reticulum,Glycoproteins,Dye,488,<NA>,522,<NA>,Alexa Fluor 488,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2773403,F13,25,NIHB127_CV384W_208,2025_05_31_NIHB127_S4,https://cellpainting-gallery.s3.us-east-1.amaz...,r06c13f25p04-ch4sk1fk1fl1,iPSC differentiated to beta cells,"Clonal line, TCF7L2/HNF4A/HNF1A::WT/WT, S4-Pan...",384,v2,...,Hoechst,Nucleus,DNA,Dye,375,<NA>,456,<NA>,33342,4
2773404,F14,13,NIHB127_CV384W_208,2025_05_31_NIHB127_S4,https://cellpainting-gallery.s3.us-east-1.amaz...,r06c14f13p04-ch4sk1fk1fl1,iPSC differentiated to beta cells,"Clonal line, TCF7L2/HNF4A/HNF1A::WT/WT, S4-Pan...",384,v2,...,Hoechst,Nucleus,DNA,Dye,375,<NA>,456,<NA>,33342,4
2773405,F14,1,NIHB127_CV384W_208,2025_05_31_NIHB127_S4,https://cellpainting-gallery.s3.us-east-1.amaz...,r06c14f01p04-ch4sk1fk1fl1,iPSC differentiated to beta cells,"Clonal line, TCF7L2/HNF4A/HNF1A::WT/WT, S4-Pan...",384,v2,...,Hoechst,Nucleus,DNA,Dye,375,<NA>,456,<NA>,33342,4
2773406,F14,14,NIHB127_CV384W_208,2025_05_31_NIHB127_S4,https://cellpainting-gallery.s3.us-east-1.amaz...,r06c14f14p04-ch4sk1fk1fl1,iPSC differentiated to beta cells,"Clonal line, TCF7L2/HNF4A/HNF1A::WT/WT, S4-Pan...",384,v2,...,Hoechst,Nucleus,DNA,Dye,375,<NA>,456,<NA>,33342,4


In [29]:
print("View of data that will be removed")
complete_df[extra_cols]

View of data that will be removed


,Z,Label,PASS_LINE_QC,stage,Plate_Map,User_Name
0,p01,ER,NaN,1,NIHB111_C384W_201,URL_OrigGlycoproteins
1,p01,ER,NaN,1,NIHB111_C384W_201,URL_OrigGlycoproteins
2,p01,ER,NaN,1,NIHB111_C384W_201,URL_OrigGlycoproteins
3,p01,ER,NaN,1,NIHB111_C384W_201,URL_OrigGlycoproteins
4,p01,ER,NaN,1,NIHB111_C384W_201,URL_OrigGlycoproteins
...,...,...,...,...,...,...
2773403,p04,DNA,True,NaN,NIHB127_CV384W_208,URL_OrigDNA
2773404,p04,DNA,True,NaN,NIHB127_CV384W_208,URL_OrigDNA
2773405,p04,DNA,True,NaN,NIHB127_CV384W_208,URL_OrigDNA
2773406,p04,DNA,True,NaN,NIHB127_CV384W_208,URL_OrigDNA


In [30]:
# After correcting any warnings above, run to save harmonization
complete_df = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df)

Removing columns not in ontology: ['Z', 'Label', 'PASS_LINE_QC', 'stage', 'Plate_Map', 'User_Name']
Adding missing ontology columns: ['Treatment_SMILES', 'Treatment_Broad_Sample', 'Treatment_InChIKey', 'Timepoint_Primary_Treatment', 'Treatment_Concentration', 'Treatment_Control_Class', 'Treatment_PubChem_CID', 'Timepoint_Secondary_Treatment', 'Timepoint_Acquisition', 'Treatment_Secondary_Treatment', 'Treatment_Solvent', 'Treatment_Mechanism']


In [31]:
saved_path = os.path.join(output_directory,f"{project_name}_harmonized_metadata_v0_1.parquet")
complete_df.to_parquet(saved_path)